# 1. Extracción (Extract)
Dado que muchos sistemas de tickets (como Zendesk o Freshdesk) exportan archivos `.xls` que en realidad son `XML Spreadsheet 2003`, usar `pd.read_excel()` nos arrojaría un error. Este bloque usa `xml.etree.ElementTree` para parsear ese falso Excel y convertirlo en un DataFrame decente.

In [ ]:
from lxml import etree
import pandas as pd
import numpy as np

# Ruta del archivo
file_path = "9000678639_tickets-April-24-2026-21_21.xls"

print("Iniciando extracción con parser tolerante a fallos (esto puede tardar unos segundos con +1.3M de líneas)...")

# El parser con recover=True ignora los errores de sintaxis del XML
parser = etree.XMLParser(recover=True, huge_tree=True)
tree = etree.parse(file_path, parser=parser)
root = tree.getroot()

# El namespace de Microsoft
ns = {'ss': 'urn:schemas-microsoft-com:office:spreadsheet'}

data = []
for row in root.findall('.//ss:Row', namespaces=ns):
    row_data = []
    for cell in row.findall('.//ss:Cell', namespaces=ns):
        data_elem = cell.find('ss:Data', namespaces=ns)
        row_data.append(data_elem.text if data_elem is not None else None)

    # Solo agregamos la fila si tiene al menos un dato (evita filas fantasma)
    if any(row_data):
        data.append(row_data)

print("XML procesado. Construyendo DataFrame...")

# Creamos el DataFrame

if len(data) > 1:
    df_raw = pd.DataFrame(data[1:], columns=data[0])
    print(f"¡Éxito! Dimensiones iniciales: {df_raw.shape}")
    display(df_raw.head(3))
else:
    print("Ocurrió un error, no se encontraron datos.")

Iniciando extracción con parser tolerante a fallos (esto puede tardar unos segundos con +1.3M de líneas)...
XML procesado. Construyendo DataFrame...
¡Éxito! Dimensiones iniciales: (10162, 45)


,ID del ticket,Asunto,Estado,Prioridad,Fuente,Tipo,Agente,Grupo,Tiempo de creación,Vencidas hasta ahora,Tiempo de resolución,Hora de cierre,Última hora de la actualización,Tiempo de respuesta inicial,Tiempo de seguimiento,Primer tiempo de respuesta (en horas),Tiempo de resolución (en horas),Interacciones de agente,Interacciones de cliente,Estado de resolución,Estado de la primera respuesta,Etiquetas,Resultados de la encuesta,Producto,Estados de todas las respuestas,Tiempo empleado,Inicio de atencion,Información de la fuente,Fin de atencion,Tipo de Solicitud,Empresa,Tipo de Programa,Origen,Origen Categoria,Dependencias,Jefe de TI que aprueba la solicitud,Estado de la aprobación,Contacto,Teléfono,Ubicación del servicio,Hora de inicio de la cita,Hora de finalización de la cita,Firma del cliente,Nombre completo,ID de contacto
0,48543,URGENTE- MAS VENTAS MAS PREMIOS,Closed,Low,Portal,Solicitud Operativa,No Agent,No Group,2024-01-02 08:55:58,2024-01-03 18:00:00,2024-01-02 09:15:36,2024-01-02 09:15:36,2024-01-02 09:15:36,2024-01-02 09:15:23,00:00,00:15:23,00:15:36,1,1,Within SLA,Within SLA,None,None,No Product,None,0.3,"2 Jan, 2024",None,"2 Jan, 2024",None,Promotick Perú,None,None,None,None,None,None,None,None,None,None,None,None,Alexia Trujillo,atrujillom@promotick.com
1,48544,Fuera de oficina Re: ANULAR TRANSACCIÓN BENEFIT,Closed,Low,Email,None,No Agent,No Group,2024-01-02 09:27:55,2024-01-04 09:27:55,2024-01-02 09:29:55,2024-01-02 09:29:55,2024-01-02 09:29:56,None,00:00,00:00:00,00:02:00,0,1,Within SLA,None,None,None,No Product,None,0.3,"2 Jan, 2024",None,"2 Jan, 2024",None,None,None,None,None,None,None,None,None,None,None,None,None,None,Francis Vargas,fvargas@promotick.com
2,48545,enviar pedido no procesado,Closed,Low,Portal,Solicitud Operativa,No Agent,No Group,2024-01-02 09:30:09,2024-01-04 09:30:09,2024-01-04 11:32:48,2024-01-04 11:32:48,2024-01-04 11:32:50,2024-01-02 10:07:11,00:00,00:37:02,20:02:39,2,1,SLA Violated,Within SLA,None,None,No Product,None,0.3,"2 Jan, 2024",None,"4 Jan, 2024",None,Promotick Perú,None,None,None,None,None,None,None,None,None,None,None,None,jassmin Ramirez,jramirez@promotick.com


# 2. Transformación (Transform) - Limpieza Estructural
Los espacios y las mayúsculas en los nombres de las columnas no son adecuados en el análisis de datos. Aquí vamos a estandarizar los nombres, eliminar columnas completamente vacías o inútiles (como 'Firma del cliente') y filtrar los datos para que coincidan con la clasificación oficial del manual de Promotick: Solo Incidentes y Requerimientos.

In [ ]:
df = df_raw.copy()
pd.set_option('display.max_columns', None)
# 1. Limpiamos los nombres de las columnas

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

# 2. Convertimos strings vacíos o con puros espacios a Nulos reales (NaN)

df = df.replace(r'^\s*$', np.nan, regex=True)

# 3. Ahora sí, borramos las columnas que son 100% nulas

df = df.dropna(axis=1, how='all')

# 4. Eliminamos columnas manuales
columnas_basura = ['firma_del_cliente', 'resultados_de_la_encuesta']
df = df.drop(columns=[col for col in columnas_basura if col in df.columns], errors='ignore')

# 5. Estandarizamos mayúsculas en los campos clave

if 'tipo' in df.columns:
    df['tipo'] = df['tipo'].astype(str).str.strip().str.capitalize()

if 'prioridad' in df.columns:
    df['prioridad'] = df['prioridad'].astype(str).str.strip().str.capitalize()

print("Columnas que quedaron:")
print(df.columns.tolist())

Columnas que quedaron:
['id_del_ticket', 'asunto', 'estado', 'prioridad', 'fuente', 'tipo', 'agente', 'grupo', 'tiempo_de_creación', 'vencidas_hasta_ahora', 'tiempo_de_resolución', 'hora_de_cierre', 'última_hora_de_la_actualización', 'tiempo_de_respuesta_inicial', 'tiempo_de_seguimiento', 'primer_tiempo_de_respuesta_en_horas', 'tiempo_de_resolución_en_horas', 'interacciones_de_agente', 'interacciones_de_cliente', 'estado_de_resolución', 'estado_de_la_primera_respuesta', 'etiquetas', 'producto', 'tiempo_empleado', 'inicio_de_atencion', 'fin_de_atencion', 'tipo_de_solicitud', 'empresa', 'tipo_de_programa', 'origen', 'origen_categoria', 'dependencias', 'jefe_de_ti_que_aprueba_la_solicitud', 'estado_de_la_aprobación', 'nombre_completo', 'id_de_contacto']


In [ ]:
df

,id_del_ticket,asunto,estado,prioridad,fuente,tipo,agente,grupo,tiempo_de_creación,vencidas_hasta_ahora,tiempo_de_resolución,hora_de_cierre,última_hora_de_la_actualización,tiempo_de_respuesta_inicial,tiempo_de_seguimiento,primer_tiempo_de_respuesta_en_horas,tiempo_de_resolución_en_horas,interacciones_de_agente,interacciones_de_cliente,estado_de_resolución,estado_de_la_primera_respuesta,producto,tiempo_empleado,inicio_de_atencion,fin_de_atencion,tipo_de_solicitud,empresa,tipo_de_programa,origen,origen_categoria,jefe_de_ti_que_aprueba_la_solicitud,estado_de_la_aprobación,nombre_completo,id_de_contacto
0,48543,URGENTE- MAS VENTAS MAS PREMIOS,Closed,Low,Portal,Solicitud operativa,No Agent,No Group,2024-01-02 08:55:58,2024-01-03 18:00:00,2024-01-02 09:15:36,2024-01-02 09:15:36,2024-01-02 09:15:36,2024-01-02 09:15:23,00:00,00:15:23,00:15:36,1,1,Within SLA,Within SLA,No Product,0.3,"2 Jan, 2024","2 Jan, 2024",None,Promotick Perú,None,None,None,None,None,Alexia Trujillo,atrujillom@promotick.com
1,48544,Fuera de oficina Re: ANULAR TRANSACCIÓN BENEFIT,Closed,Low,Email,None,No Agent,No Group,2024-01-02 09:27:55,2024-01-04 09:27:55,2024-01-02 09:29:55,2024-01-02 09:29:55,2024-01-02 09:29:56,None,00:00,00:00:00,00:02:00,0,1,Within SLA,None,No Product,0.3,"2 Jan, 2024","2 Jan, 2024",None,None,None,None,None,None,None,Francis Vargas,fvargas@promotick.com
2,48545,enviar pedido no procesado,Closed,Low,Portal,Solicitud operativa,No Agent,No Group,2024-01-02 09:30:09,2024-01-04 09:30:09,2024-01-04 11:32:48,2024-01-04 11:32:48,2024-01-04 11:32:50,2024-01-02 10:07:11,00:00,00:37:02,20:02:39,2,1,SLA Violated,Within SLA,No Product,0.3,"2 Jan, 2024","4 Jan, 2024",None,Promotick Perú,None,None,None,None,None,jassmin Ramirez,jramirez@promotick.com
3,48546,Curso Online: Reducción de Impuesto a la Renta...,Closed,Low,Email,None,No Agent,No Group,2024-01-02 10:11:35,2024-01-04 10:11:35,2024-01-02 10:18:10,2024-01-02 10:18:10,2024-01-02 10:18:10,None,00:00,00:00:00,00:06:35,0,1,Within SLA,None,No Product,0.3,"2 Jan, 2024","2 Jan, 2024",None,None,None,None,None,None,None,Con Plantillas Excel,desarrollo@amb20online.com
4,48547,Copia en correos Canje en ClubOlimpo EC,Closed,Low,Portal,Solicitud operativa,No Agent,No Group,2024-01-02 10:52:01,2024-01-04 10:52:01,2024-01-04 14:09:37,2024-01-04 14:09:37,2024-01-04 14:09:37,2024-01-02 13:35:22,00:00,02:43:21,21:17:36,3,5,SLA Violated,Within SLA,No Product,0.3,"4 Jan, 2024","4 Jan, 2024",None,Promotick Ecuador,None,None,None,None,None,GRACE ESTRELLA,gestrella@promotick.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2445,51576,Socio Tuko - Verificar que los catálogos estén...,Closed,Low,Portal,Solicitud operativa,No Agent,No Group,2024-06-12 16:08:08,2024-06-14 16:08:08,2024-06-13 10:21:36,2024-06-13 10:21:36,2024-06-13 10:21:37,2024-06-13 10:21:22,00:00,03:13:14,03:13:28,1,1,Within SLA,Within SLA,No Product,0.3,"13 Jun, 2024","13 Jun, 2024",None,Promotick Ecuador,Otros,None,None,None,None,Helen Torres,htorres@promotick.com
2446,51577,Asismiles - carga usuarios,Closed,Low,Portal,Solicitud operativa,No Agent,No Group,2024-06-12 16:49:48,2024-06-14 16:49:48,2024-06-14 08:58:00,2024-06-14 08:58:00,2024-06-14 08:58:00,2024-06-13 09:52:33,00:00,02:02:45,10:10:12,1,2,Within SLA,Within SLA,No Product,None,None,None,None,Motiva Ecuador,Puntos,None,None,None,None,Andrés Agurto,aagurto@motivasoluciones.com
2447,51578,ACTUALIZACIONES REPSOL,Closed,Low,Portal,Solicitud operativa,No Agent,No Group,2024-06-12 16:55:04,2024-06-14 16:55:04,2024-06-14 14:27:55,2024-06-14 14:27:55,2024-06-14 14:27:55,2024-06-12 17:20:46,00:00,00:25:42,15:32:51,2,4,Within SLA,Within SLA,No Product,0.3,"14 Jun, 2024","14 Jun, 2024",None,Promotick Perú,Otros,None,None,None,None,Carolina Pastor,cpastor@promotick.com
2448,51579,Correlativo plantilla Enjoy,Closed,Low,Portal,Plantilla enjoy,No Agent,No Group,2024-06-13 09:29:22,2024-06-17 09:29:23,2024-06-17 09

# 3. Transformación - Tiempos y Fechas
Para calcular los indicadores operacionales como "Tiempo Promedio de Atención" o "Tiempo de Primera Respuesta", necesitamos que los strings de fecha sean objetos `datetime`.
Convertiremos las fechas clave y llenaremos los vacíos lógicos.

In [ ]:
# Convertimos las columnas de fechas principales
cols_fechas = ['tiempo_de_creación', 'tiempo_de_resolución', 'hora_de_cierre', 'última_hora_de_la_actualización']

for col in cols_fechas:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Tiempo desde que se creó hasta que se resolvió (en horas)
if 'tiempo_de_resolución' in df.columns and 'tiempo_de_creación' in df.columns:
    df['lead_time_horas'] = ((df['tiempo_de_resolución'] - df['tiempo_de_creación']).dt.total_seconds() / 3600).round(4)

# Bandera para saber si el ticket sigue abierto (Backlog)
# Según el manual, backlog es la diferencia entre abiertos y cerrados.
df['esta_abierto'] = df['estado'].apply(lambda x: 1 if str(x).lower() not in ['cerrado', 'resuelto', 'closed', 'resolved'] else 0)

df[['tiempo_de_creación', 'tiempo_de_resolución', 'lead_time_horas', 'esta_abierto']].head()

,tiempo_de_creación,tiempo_de_resolución,lead_time_horas,esta_abierto
0,2024-01-02 08:55:58,2024-01-02 09:15:36,0.3272,0
1,2024-01-02 09:27:55,2024-01-02 09:29:55,0.0333,0
2,2024-01-02 09:30:09,2024-01-04 11:32:48,50.0442,0
3,2024-01-02 10:11:35,2024-01-02 10:18:10,0.1097,0
4,2024-01-02 10:52:01,2024-01-04 14:09:37,51.2933,0


# 4. Enriquecimiento de Datos y KPIs
De acuerdo con los "Indicadores de Gestión" del manual, necesitamos poder medir el Cumplimiento de SLA y el Backlog Crítico (Tickets de alta prioridad pendientes).
Aquí creamos esas métricas para que cuando se use Seaborn o Plotly, los gráficos sean fáciles de realizar.

In [ ]:
# Backlog Crítico: Alta prioridad + Abierto

if 'prioridad' in df.columns:
    df['backlog_critico'] = np.where((df['esta_abierto'] == 1) & (df['prioridad'] == 'Alta'), 1, 0)

# Cumplimiento de SLA (asumiendo que 'vencidas_hasta_ahora' es un booleano o indicador del sistema)
# Si no existe, usamos 'estado_de_resolución'
if 'estado_de_resolución' in df.columns:
    df['incumple_sla'] = df['estado_de_resolución'].apply(lambda x: 1 if str(x).lower() in ['sla violated'] else 0)
    df['cumple_sla'] = 1 - df['incumple_sla']

# Extraemos el mes y el día de la semana de la creación
df['mes_creacion'] = df['tiempo_de_creación'].dt.to_period('M')
df['dia_semana_creacion'] = df['tiempo_de_creación'].dt.day_name()

print(f"Total de Backlog Crítico actual: {df['backlog_critico'].sum()}")

Total de Backlog Crítico actual: 0


# 5.  Validación y limpieza final del dataset

Después de la creación de variables, se realiza una última etapa de limpieza del dataset con el objetivo de eliminar variables que no aportan valor analítico o que han sido descontinuadas según lo indicado por el representante de Promotick. Esto permite reducir ruido en los datos y mantener únicamente información relevante para el análisis.

Se eliminan las siguientes variables:

- "tiempo_de_seguimiento": no utilizada en los procesos actuales.
- "estado_de_la_aprobación": campo asociado a aprobaciones no relevante  para el análisis.
- "jefe_de_ti_que_aprueba_la_solicitud": variable administrativa sin valor analítico.
- "producto": columna sin información (vacía).
- "id_de_contacto": identificador no necesario para el análisis de negocio.

In [ ]:
df.drop(columns=[ "tiempo_de_seguimiento", "estado_de_la_aprobación", "jefe_de_ti_que_aprueba_la_solicitud","producto","id_de_contacto"],
        inplace=True)

# 6. Asignación de tipos de datos
En esta etapa se asigna a cada variable el tipo de dato más adecuado según su naturaleza (numérico, categórico, fecha o duración). Esto permite asegurar la correcta interpretación de los datos y su uso en análisis posteriores.

In [ ]:
# Columnas numericas

cols_numericas = [
    'interacciones_de_agente',
    'interacciones_de_cliente',
    'tiempo_empleado',
    'lead_time_horas'
]

for col in cols_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fechas

cols_fechas = [
    'inicio_de_atencion',
    'fin_de_atencion',
    'tiempo_de_respuesta_inicial',
    'tiempo_de_creación',
    'tiempo_de_resolución',
    'hora_de_cierre',
    'última_hora_de_la_actualización',
    'vencidas_hasta_ahora'
]

for col in cols_fechas:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

cols_duracion = [
    'primer_tiempo_de_respuesta_en_horas',
    'tiempo_de_resolución_en_horas'
]

for col in cols_duracion:
    if col in df.columns:
        df[col] = pd.to_timedelta(df[col], errors='coerce')



# Columnas binarias

cols_sla = [
    'esta_abierto',
    'backlog_critico',
    'incumple_sla',
    'cumple_sla'
]

for col in cols_sla:
    if col in df.columns:
        df[col] = df[col].astype('int')

cols_excluir = [
    'id_del_ticket',
    'id_de_contacto',
    'asunto',
    'nombre_completo',
    'etiquetas',
    'dependencias'
]

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10162 entries, 0 to 10161
Data columns (total 43 columns):
 #   Column                               Non-Null Count  Dtype          
---  ------                               --------------  -----          
 0   id_del_ticket                        10162 non-null  object         
 1   asunto                               10161 non-null  object         
 2   estado                               10162 non-null  object         
 3   prioridad                            10162 non-null  object         
 4   fuente                               10162 non-null  object         
 5   tipo                                 10162 non-null  object         
 6   agente                               10162 non-null  object         
 7   grupo                                10162 non-null  object         
 8   tiempo_de_creación                   10162 non-null  datetime64[ns] 
 9   vencidas_hasta_ahora                 10162 non-null  datetime64[ns] 
 10

# 7. Carga (Load) - Exportar la Data Limpia
La limpieza está lista. Guardemos esto como un `.csv`. Con esto, la data ya está lista para usarse.

In [ ]:
# Guardamos en un archivo CSV limpio, separado por comas
clean_file_name = "tickets_promotick_clean.csv"
df.to_csv(clean_file_name, index=False, encoding='utf-8')

print(f"¡Éxito! Datos limpios exportados a {clean_file_name}")
print("Aquí un resumen de tu dataset final:")
df.info()

¡Éxito! Datos limpios exportados a tickets_promotick_clean.csv
Aquí un resumen de tu dataset final:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10162 entries, 0 to 10161
Data columns (total 38 columns):
 #   Column                               Non-Null Count  Dtype          
---  ------                               --------------  -----          
 0   id_del_ticket                        10162 non-null  object         
 1   asunto                               10161 non-null  object         
 2   estado                               10162 non-null  object         
 3   prioridad                            10162 non-null  object         
 4   fuente                               10162 non-null  object         
 5   tipo                                 10162 non-null  object         
 6   agente                               10162 non-null  object         
 7   grupo                                10162 non-null  object         
 8   tiempo_de_creación                   10162 non

In [ ]:
df

,id_del_ticket,asunto,estado,prioridad,fuente,tipo,agente,grupo,tiempo_de_creación,vencidas_hasta_ahora,tiempo_de_resolución,hora_de_cierre,última_hora_de_la_actualización,tiempo_de_respuesta_inicial,primer_tiempo_de_respuesta_en_horas,tiempo_de_resolución_en_horas,interacciones_de_agente,interacciones_de_cliente,estado_de_resolución,estado_de_la_primera_respuesta,etiquetas,tiempo_empleado,inicio_de_atencion,fin_de_atencion,tipo_de_solicitud,empresa,tipo_de_programa,origen,origen_categoria,dependencias,nombre_completo,lead_time_horas,esta_abierto,backlog_critico,incumple_sla,cumple_sla,mes_creacion,dia_semana_creacion
0,48543,URGENTE- MAS VENTAS MAS PREMIOS,Closed,Low,Portal,Solicitud operativa,No Agent,No Group,2024-01-02 08:55:58,2024-01-03 18:00:00,2024-01-02 09:15:36,2024-01-02 09:15:36,2024-01-02 09:15:36,2024-01-02 09:15:23,0 days 00:15:23,0 days 00:15:36,1,1,Within SLA,Within SLA,None,0.3,2024-01-02,2024-01-02,None,Promotick Perú,None,None,None,None,Alexia Trujillo,0.3272,0,0,0,1,2024-01,Tuesday
1,48544,Fuera de oficina Re: ANULAR TRANSACCIÓN BENEFIT,Closed,Low,Email,None,No Agent,No Group,2024-01-02 09:27:55,2024-01-04 09:27:55,2024-01-02 09:29:55,2024-01-02 09:29:55,2024-01-02 09:29:56,NaT,0 days 00:00:00,0 days 00:02:00,0,1,Within SLA,None,None,0.3,2024-01-02,2024-01-02,None,None,None,None,None,None,Francis Vargas,0.0333,0,0,0,1,2024-01,Tuesday
2,48545,enviar pedido no procesado,Closed,Low,Portal,Solicitud operativa,No Agent,No Group,2024-01-02 09:30:09,2024-01-04 09:30:09,2024-01-04 11:32:48,2024-01-04 11:32:48,2024-01-04 11:32:50,2024-01-02 10:07:11,0 days 00:37:02,0 days 20:02:39,2,1,SLA Violated,Within SLA,None,0.3,2024-01-02,2024-01-04,None,Promotick Perú,None,None,None,None,jassmin Ramirez,50.0442,0,0,1,0,2024-01,Tuesday
3,48546,Curso Online: Reducción de Impuesto a la Renta...,Closed,Low,Email,None,No Agent,No Group,2024-01-02 10:11:35,2024-01-04 10:11:35,2024-01-02 10:18:10,2024-01-02 10:18:10,2024-01-02 10:18:10,NaT,0 days 00:00:00,0 days 00:06:35,0,1,Within SLA,None,None,0.3,2024-01-02,2024-01-02,None,None,None,None,None,None,Con Plantillas Excel,0.1097,0,0,0,1,2024-01,Tuesday
4,48547,Copia en correos Canje en ClubOlimpo EC,Closed,Low,Portal,Solicitud operativa,No Agent,No Group,2024-01-02 10:52:01,2024-01-04 10:52:01,2024-01-04 14:09:37,2024-01-04 14:09:37,2024-01-04 14:09:37,2024-01-02 13:35:22,0 days 02:43:21,0 days 21:17:36,3,5,SLA Violated,Within SLA,None,0.3,2024-01-04,2024-01-04,None,Promotick Ecuador,None,None,None,None,GRACE ESTRELLA,51.2933,0,0,1,0,2024-01,Tuesday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10157,59320,SOLICITUD REPORTE PEDIOS BCP ENALTA CON FECHA ...,Closed,High,Portal,Requerimiento de servicio,Sebastián Fabrizio Alvino Linares,Centro de Servicios TI,2025-12-31 10:11:56,2026-01-02 10:11:56,2026-01-02 16:00:05,2026-01-02 16:00:05,2026-01-02 16:05:22,2026-01-02 13:54:16,0 days 21:42:20,0 days 23:48:09,1,2,SLA Violated,SLA Violated,None,0.2,2026-01-02,2026-01-02,Reportes,Promotick Perú,Beneficios,Reportes y Consultas,Reportes,Infraestructura,Junior Morales,53.8025,0,0,1,0,2025-12,Wednesday
10158,59321,CLIENTE BCP ENALTA - CAMBIO DE DIRECCION,Closed,High,Portal,Incidente del servicio,Sebastián Fabrizio Alvino Linares,Centro de Servicios TI,2025-12-31 10:23:38,2026-01-02 10:23:38,2026-01-05 09:25:37,2026-01-05 09:25:37,2026-01-05 09:25:37,2026-01-02 17:46:49,1 days 01:23:11,1 days 02:01:59,2,1,SLA Violated,SLA Violated,None,0.1,2026-01-02,2026-01-02,Soporte Técnico,Promotick Perú,Beneficios,Operaciones de Cambios en Producción,Actualización,Infraestructura,Maribel Paz,119.0331,0,0,1,0,2025-12,Wednesday
10159,59322,CLIENTE BCP ENALTA - Cambio de direccion,Closed,High,Portal,Incidente del servicio,Sebastián Fabrizio Alvino Linares,Centro de Servicios TI,2025-12-31 10:46:12,2026-01-02 10:46:12,2026-01-05 09:25:57,2026-01-05 09:25:57,2026-01-05 09:26:00,2026-01-02 17:48:10,1 day

# 8. Creamos el diccionario de datos

In [ ]:
diccionario = []

if 'id_del_ticket' in df.columns:
    diccionario.append({
        "variable": "id_del_ticket",
        "tipo": "Identificador",
        "min_id": df['id_del_ticket'].min(),
        "max_id": df['id_del_ticket'].max(),
        "n_unicos": df['id_del_ticket'].nunique()
    })


# Numéricas

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

num_cols = [c for c in num_cols if c not in ['esta_abierto', 'backlog_critico', 'incumple_sla', 'cumple_sla']]

for col in num_cols:
    diccionario.append({
        "variable": col,
        "tipo": "Numérico",
        "min": df[col].min(),
        "max": df[col].max(),
        "valores_unicos": df[col].nunique()
    })


# 2. FECHAS
date_cols = df.select_dtypes(include=['datetime64[ns]']).columns

for col in date_cols:
    diccionario.append({
        "variable": col,
        "tipo": "Fecha",
        "min_fecha": df[col].min(),
        "max_fecha": df[col].max()
    })


# 3. CATEGÓRICAS
cat_cols = df.select_dtypes(include=['object']).columns

cat_cols = [c for c in cat_cols if c != 'id_del_ticket']

for col in cat_cols:
    diccionario.append({
        "variable": col,
        "tipo": "Categórico",
        "n_valores": df[col].nunique(),
        "valores": df[col].dropna().unique()[:10]
    })


df_diccionario = pd.DataFrame(diccionario)
df_diccionario

,variable,tipo,min_id,max_id,n_unicos,min,max,valores_unicos,min_fecha,max_fecha,n_valores,valores
0,id_del_ticket,Identificador,48543,59324,10162.0,NaN,NaN,NaN,NaT,NaT,NaN,NaN
1,interacciones_de_agente,Numérico,NaN,NaN,NaN,0.0,21.0000,21.0,NaT,NaT,NaN,NaN
2,interacciones_de_cliente,Numérico,NaN,NaN,NaN,1.0,29.0000,23.0,NaT,NaT,NaN,NaN
3,tiempo_empleado,Numérico,NaN,NaN,NaN,-2.0,60.0000,48.0,NaT,NaT,NaN,NaN
4,lead_time_horas,Numérico,NaN,NaN,NaN,0.0,3746.0406,9587.0,NaT,NaT,NaN,NaN
5,tiempo_de_creación,Fecha,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-02 08:55:58,2025-12-31 13:42:48,NaN,NaN
6,vencidas_hasta_ahora,Fecha,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-03 18:00:00,2026-02-02 17:52:02,NaN,NaN
7,tiempo_de_resolución,Fecha,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-02 09:15:36,2026-04-22 11:36:00,NaN,NaN
8,hora_de_cierre,Fecha,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-02 09:15:36,2026-04-22 11:36:00,NaN,NaN
9,última_hora_de_la_actualización,Fecha,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-02 09:15:36,2026-04-22 11:36:00,NaN,NaN
